# 🏦 Bank Binary Classification — Dengesiz Veri ile İkili Sınıflandırma

## 🎯 Problem
Banka müşterisi verilerinden **müşterinin ürüne abone olup olmayacağını** tahmin ediyoruz.
Veri seti dengesizdir: %88 hayır, %12 evet → Accuracy tek başına yanıltıcıdır.

## 💡 Bu notebook'tan ne öğrenirsin?
- sklearn Pipeline + ColumnTransformer ile uçtan uca sızıntısız iş akışı
- Dengesiz veri yönetimi: class_weight ve scale_pos_weight
- GridSearchCV ile F1 skoruna göre hiperparametre optimizasyonu
- Accuracy vs Precision / Recall / F1: dengesiz veride neden Accuracy yetmez?
- RandomForest ve XGBoost karşılaştırması

## 📊 Veri Seti
- **Kaynak:** [Kaggle - Playground Series S5E8](https://www.kaggle.com/competitions/playground-series-s5e8)
- **Boyut:** ~750.000 satır
- **Sınıf dağılımı:** 0 (%87.9) | 1 (%12.1) — Dengesiz!

## 1. Kütüphaneler ve Veri Yükleme

Temel veri işleme kütüphanelerini (`numpy`, `pandas`) içe aktarıyor ve Kaggle çalışma ortamındaki eğitim (`train.csv`) ile test (`test.csv`) veri setlerini yüklüyoruz.

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/playground-series-s5e8/sample_submission.csv
/kaggle/input/competitions/playground-series-s5e8/train.csv
/kaggle/input/competitions/playground-series-s5e8/test.csv


In [2]:
train=pd.read_csv((__import__("glob").glob("/kaggle/input/**/train.csv", recursive=True)[0] if not os.path.exists("/kaggle/input/playground-series-s5e8/train.csv") else "/kaggle/input/playground-series-s5e8/train.csv"))
train.head()

,id,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,0,42,technician,married,secondary,no,7,no,no,cellular,25,aug,117,3,-1,0,unknown,0
1,1,38,blue-collar,married,secondary,no,514,no,no,unknown,18,jun,185,1,-1,0,unknown,0
2,2,36,blue-collar,married,secondary,no,602,yes,no,unknown,14,may,111,2,-1,0,unknown,0
3,3,27,student,single,secondary,no,34,yes,no,unknown,28,may,10,2,-1,0,unknown,0
4,4,26,technician,married,secondary,no,889,yes,no,cellular,3,feb,902,1,-1,0,unknown,1


In [3]:
test=pd.read_csv((__import__("glob").glob("/kaggle/input/**/test.csv", recursive=True)[0] if not os.path.exists("/kaggle/input/playground-series-s5e8/test.csv") else "/kaggle/input/playground-series-s5e8/test.csv"))

## 2. Veri İnceleme ve Özellik/Hedef Ayrımı

Modelleme öncesinde `id` sütununu kaldırıyor, hedef değişken (`y`) ile özellikleri (`x`) ayırıyoruz. Ardından eksik değer kontrolü ve veri tiplerini (`int64`, `object`) inceliyoruz.

In [4]:
from sklearn.model_selection import train_test_split

train=train.drop("id",axis=1)

y=train["y"]
x=train.drop("y",axis=1)

In [5]:
train.isnull().sum()

age          0
job          0
marital      0
education    0
default      0
balance      0
housing      0
loan         0
contact      0
day          0
month        0
duration     0
campaign     0
pdays        0
previous     0
poutcome     0
y            0
dtype: int64

In [6]:
x.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 750000 entries, 0 to 749999
Data columns (total 16 columns):
 #   Column     Non-Null Count   Dtype 
---  ------     --------------   ----- 
 0   age        750000 non-null  int64 
 1   job        750000 non-null  object
 2   marital    750000 non-null  object
 3   education  750000 non-null  object
 4   default    750000 non-null  object
 5   balance    750000 non-null  int64 
 6   housing    750000 non-null  object
 7   loan       750000 non-null  object
 8   contact    750000 non-null  object
 9   day        750000 non-null  int64 
 10  month      750000 non-null  object
 11  duration   750000 non-null  int64 
 12  campaign   750000 non-null  int64 
 13  pdays      750000 non-null  int64 
 14  previous   750000 non-null  int64 
 15  poutcome   750000 non-null  object
dtypes: int64(7), object(9)
memory usage: 91.6+ MB


In [7]:
y.info()

<class 'pandas.core.series.Series'>
RangeIndex: 750000 entries, 0 to 749999
Series name: y
Non-Null Count   Dtype
--------------   -----
750000 non-null  int64
dtypes: int64(1)
memory usage: 5.7 MB


## 3. Ön İşleme Pipeline'ı ve RandomForest (Baseline Model)

Veri setinde hem sayısal hem de kategorik (`job`, `marital`, `education`, vb.) sütunlar bulunmaktadır.
- `ColumnTransformer` + `OneHotEncoder(handle_unknown="ignore")` ile kategorik değişkenleri pipeline içinde dönüştürüyoruz; böylece `train/test` veri sızıntısını (data leakage) önlüyoruz.
- Dengesiz sınıf yapısını ele almak için `RandomForestClassifier` içinde `class_weight='balanced'` parametresini kullanıyoruz.

In [8]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier


cat_cols=list(x.select_dtypes(include="object").columns)

cat_transformers=Pipeline(steps=[
    ("onehotecode",OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    
    ("cat", cat_transformers, cat_cols)
    
], remainder="passthrough") 

model=Pipeline(steps=[
    ("preprocessor",preprocessor),
    ("model",RandomForestClassifier(n_estimators=200,max_depth=4,class_weight='balanced'))
])

x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)

model.fit(x_train,y_train)

/usr/local/lib/python3.13/dist-packages/sklearn/compose/_column_transformer.py:1667: FutureWarning: 
The format of the columns of the 'remainder' transformer in ColumnTransformer.transformers_ will change in version 1.7 to match the format of the other transformers.
At the moment the remainder columns are stored as indices (of type int). With the same ColumnTransformer configuration, in the future they will be stored as column names (of type str).
To use the new behavior now and suppress this warning, use ColumnTransformer(force_int_remainder_cols=False).

  warnings.warn(


Pipeline(steps=[('preprocessor',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('cat',
                                                  Pipeline(steps=[('onehotecode',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['job', 'marital',
                                                   'education', 'default',
                                                   'housing', 'loan', 'contact',
                                                   'month', 'poutcome'])])),
                ('model',
                 RandomForestClassifier(class_weight='balanced', max_depth=4,
                                        n_estimators=200))])

### RandomForest Başarı Değerlendirmesi

Eğitim ve test doğruluk (accuracy) skorlarını karşılaştırarak modelin genelleme performansını kontrol ediyoruz.

In [9]:
print(model.score(x_train,y_train))
print(model.score(x_test,y_test))

0.84138
0.8421866666666666


## 4. XGBoost ile Sınıflandırma ve Dengesiz Veri Yönetimi

### ⚠️ Kritik: scale_pos_weight Neden y_train'den Hesaplanmalı?

`scale_pos_weight = negatif / pozitif` değeri XGBoost'a "pozitif örneklere X kat daha fazla ağırlık ver" der.
Eğer bunu **test verisinden** hesaplarsak → test verisindeki dağılım bilgisi train'e sızmış olur (subtle leakage).
Her zaman `y_train`'den hesapla.

In [10]:
from xgboost import XGBClassifier

# scale_pos_weight = negatif_sınıf / pozitif_sınıf (SADECE y_train'den hesapla!)
neg_count = (y_train == 0).sum()
pos_count = (y_train == 1).sum()
scale_pos_weight = neg_count / pos_count
print(f"scale_pos_weight: {scale_pos_weight:.2f} ({neg_count} negatif / {pos_count} pozitif)")

model1=Pipeline(steps=[
    ("preprocessor",preprocessor),
    ("model",XGBClassifier(scale_pos_weight=scale_pos_weight, use_label_encoder=False, eval_metric='logloss'))
])

model1.fit(x_train,y_train)

scale_pos_weight: 7.30 (527717 negatif / 72283 pozitif)


/usr/local/lib/python3.13/dist-packages/xgboost/training.py:200: UserWarning: [13:21:01] WARNING: /__w/xgboost/xgboost/src/learner.cc:794: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.13/dist-packages/sklearn/compose/_column_transformer.py:1667: FutureWarning: 
The format of the columns of the 'remainder' transformer in ColumnTransformer.transformers_ will change in version 1.7 to match the format of the other transformers.
At the moment the remainder columns are stored as indices (of type int). With the same ColumnTransformer configuration, in the future they will be stored as column names (of type str).
To use the new behavior now and suppress this warning, use ColumnTransformer(force_int_remainder_cols=False).

  warnings.warn(


Pipeline(steps=[('preprocessor',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('cat',
                                                  Pipeline(steps=[('onehotecode',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['job', 'marital',
                                                   'education', 'default',
                                                   'housing', 'loan', 'contact',
                                                   'month', 'poutcome'])])),
                ('model',
                 XGBClassifier(base_score=None, booster=None, callbacks=None,
                               colsample_bylevel=None, co...
                               feature_types=None, feature_weights=None,
                               gamma=None, grow_policy=None,
                               importance_type=None,
                               interaction_constraints=None, learning_rate=None,
                               max_bin=None, max_cat_threshold=None,
                               max_cat_to_onehot=None, max_delta_step=None,
                               max_depth=None, max_leaves=None,
                               min_child_weight=None, missing=nan,
                               monotone_constraints=None, multi_strategy=None,
                               n_estimators=None, n_jobs=None,
                               num_parallel_tree=None, ...))])

### XGBoost Değerlendirmesi: Accuracy vs Precision / Recall / F1

Dengesiz veri setlerinde yalnızca `accuracy` skoruna bakmak yanıltıcı olabilir. Sınıf oranlarını inceleyip `classification_report` ile özellikle azınlık sınıfı (class `1`) için **precision**, **recall** ve **F1-score** değerlerini değerlendiriyoruz.

In [11]:
print(model1.score(x_train,y_train))
print(model1.score(x_test,y_test))

0.8910566666666667
0.8869933333333333


In [12]:
y.value_counts(normalize=True)

y
0    0.879349
1    0.120651
Name: proportion, dtype: float64

In [13]:
from sklearn.metrics import classification_report

y_pred=model1.predict(x_test)

classification_report(y_test, y_pred)


'              precision    recall  f1-score   support\n\n           0       0.99      0.88      0.93    131795\n           1       0.52      0.93      0.67     18205\n\n    accuracy                           0.89    150000\n   macro avg       0.75      0.91      0.80    150000\nweighted avg       0.93      0.89      0.90    150000\n'

### Manuel Hiperparametre Denemesi (XGBoost)

GridSearch öncesinde `learning_rate=0.1`, `max_depth=7` ve `n_estimators=200` hiperparametreleriyle modelin performansını gözlemliyoruz.

In [14]:
model1=Pipeline(steps=[
    ("preprocessor",preprocessor),
    ("model",XGBClassifier(learning_rate= 0.1, max_depth= 7,n_estimators= 200,scale_pos_weight=scale_pos_weight, use_label_encoder=False, eval_metric='logloss'))
])

model1.fit(x_train,y_train)

print(model1.score(x_train,y_train))
print(model1.score(x_test,y_test))

/usr/local/lib/python3.13/dist-packages/xgboost/training.py:200: UserWarning: [13:21:12] WARNING: /__w/xgboost/xgboost/src/learner.cc:794: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


0.89154
0.887


## 5. GridSearchCV ile Hiperparametre Optimizasyonu (F1 Odaklı)

Dengesiz veri problemlerinde model seçimi için `scoring='f1'` metriğini kullanmak, azınlık sınıfı performansını doğrudan optimize etmemizi sağlar. `3-fold` çapraz doğrulama (`cv=3`) ile en iyi `max_depth`, `learning_rate` ve `n_estimators` kombinasyonunu arıyoruz.

In [15]:
from sklearn.model_selection import GridSearchCV


param_grid = {
    'model__max_depth': [3, 5, 7],          
    'model__learning_rate': [0.01, 0.1],   
    'model__n_estimators': [100, 200]}      


grid_search = GridSearchCV(model1, param_grid=param_grid, cv=3, scoring='f1', verbose=1)

grid_search.fit(x_train, y_train)


print("En iyi parametreler:", grid_search.best_params_)


best_model = grid_search.best_estimator_
y_pred = best_model.predict(x_test)

print(classification_report(y_test, y_pred))

Fitting 3 folds for each of 12 candidates, totalling 36 fits


/usr/local/lib/python3.13/dist-packages/xgboost/training.py:200: UserWarning: [13:21:28] WARNING: /__w/xgboost/xgboost/src/learner.cc:794: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.13/dist-packages/xgboost/training.py:200: UserWarning: [13:21:33] WARNING: /__w/xgboost/xgboost/src/learner.cc:794: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.13/dist-packages/xgboost/training.py:200: UserWarning: [13:21:39] WARNING: /__w/xgboost/xgboost/src/learner.cc:794: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.13/dist-packages/xgboost/training.py:200: UserWarning: [13:21:44] WARNING: /__w/xgboost/xgboost/src/learner.cc:794: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.13/dist-packages/xgboost/training.py:

En iyi parametreler: {'model__learning_rate': 0.1, 'model__max_depth': 7, 'model__n_estimators': 200}
              precision    recall  f1-score   support

           0       0.99      0.88      0.93    131795
           1       0.52      0.93      0.67     18205

    accuracy                           0.89    150000
   macro avg       0.75      0.91      0.80    150000
weighted avg       0.93      0.89      0.90    150000



In [16]:
print("En iyi parametreler:", grid_search.best_params_)
print("En iyi F1 skoru:", grid_search.best_score_)

En iyi parametreler: {'model__learning_rate': 0.1, 'model__max_depth': 7, 'model__n_estimators': 200}
En iyi F1 skoru: 0.6667868834980789


## 6. Test Tahminleri ve Kaggle Submission Oluşturma

En iyi model (`best_model`) ile `test` veri seti üzerinde tahmin üretiyor ve `submission.csv` dosyasını kaydediyoruz.

In [17]:
submission=pd.read_csv((__import__("glob").glob("/kaggle/input/**/sample_submission.csv", recursive=True)[0] if not os.path.exists("/kaggle/input/playground-series-s5e8/sample_submission.csv") else "/kaggle/input/playground-series-s5e8/sample_submission.csv"))
submission.head()

,id,y
0,750000,0.5
1,750001,0.5
2,750002,0.5
3,750003,0.5
4,750004,0.5


In [18]:
y_pred1 = best_model.predict(test.drop("id", axis=1))       
submission["y"] = y_pred1
submission.to_csv("submission.csv", index=False)

## 📈 Sonuçlar Özeti (Holdout Doğrulama — %20 `train_test_split`)

> **Not:** Tüm değerlendirme skorları yarışma `test.csv` üzerinde değil, `train.csv` içinden ayrılan %20 holdout doğrulama seti (`150.000` satır: `131.795` sınıf `0`, `18.205` sınıf `1`) üzerinde ölçülmüştür.
>
> **Majority Baseline:** Her gözleme `0` diyen model zaten **%87.86 Holdout Accuracy** (`0.00` Class 1 F1) alır; bu nedenle ana metrik **Class 1 F1** skorudur.

| Model | **Class 1 F1 (Ana Metrik)** | Holdout Accuracy | Yöntem / Durum |
|---|---|---|---|
| Majority Baseline | **`0.00`** | `%87.86` | Her gözleme `0` tahmini |
| RandomForest (balanced) | Hesaplanmadı | `%84.22` (`0.8422`) | `class_weight='balanced'` |
| XGBoost (`scale_pos_weight`) | **`0.67`** | `%88.70` (`0.8870`) | `scale_pos_weight = 7.30` (`y_train`'den) |
| XGBoost + GridSearchCV | **`0.67`** *(3-Fold CV F1: **`0.6668`**)* | `%88.70` (`0.8870`) | `learning_rate=0.1, max_depth=7, n_estimators=200` (Class 1 Precision `0.52`, Recall `0.93`) |

> ⚠️ **`duration` Sızıntı Riski:** `duration` (son görüşme süresi) yalnızca arama bittikten sonra bilindiği için gerçek dünyada tahmin anında bilinemez (*look-ahead leakage*). Bu çalışmada Kaggle yarışma şemasına sadık kalınarak benchmark amacıyla korunmuştur.

## 🔑 Öğrenilen Dersler
1. **Dengesiz veride Accuracy yanıltır:** Majority baseline %87.86 iken asıl kazanım Class 1 F1'dir (`0.00` → `0.67`)
2. **F1 ile optimize et:** `GridSearchCV`'de `scoring='f1'` daha anlamlıdır
3. **`scale_pos_weight` `y_train`'den hesaplanır, asla `y_test`'ten değil** (`527717 / 72283 = 7.30`)
4. **Pipeline zorunludur:** `OneHotEncoder`'ı pipeline dışına alırsanız holdout kategorileri train'e sızar
5. **Eğitim ve test şeması uyumlu olmalıdır:** `train`'den çıkarılan `id` sütunu tahmin sırasında `test`'ten de çıkarılmalıdır